# ✈️ Flight Option Finder — Agentic AI Demo

**Project T10** — CSE476 AI Agent Assignment

This notebook demonstrates a **real AI agent** (not a chatbot) that:
1. Searches for flights using `search_flights(from, to)`
2. Compares prices using `compare_price(options, budget)`
3. Remembers user preferences across turns
4. Makes decisions based on budget, non-stop preference, and price
5. Always provides a second-best fallback


## 1. Setup & Imports


In [1]:
import sys, os

# Robust path discovery: locate project root containing 'src/agent/flight_agent.py'
current = os.path.abspath(os.getcwd())
candidates = [
    current,
    os.path.abspath(os.path.join(current, "..")),
    os.path.abspath(os.path.join(current, "CSE476")),
    os.path.abspath(os.path.join(current, "..", "CSE476")),
    r"c:\Users\phani\Downloads\Agentic-ai project\CSE476",
]

project_root = None
for p in candidates:
    if os.path.exists(os.path.join(p, "src", "agent", "flight_agent.py")):
        project_root = p
        if p not in sys.path:
            sys.path.insert(0, p)
        break

from src.agent.flight_agent import FlightAgent
from src.agent.memory import ConversationMemory
from src.tools.search_flights import search_flights
from src.tools.compare_price import compare_price

print("✅ All imports successful")
print(f"   Project root: {project_root}")


✅ All imports successful
   Project root: C:\Users\phani\Downloads\Agentic-ai project\CSE476


## 2. Agent Execution Trace Helper

This renders the agent's multi-step **Plan → Act → Observe → Decide** execution trace as a formatted table with badges for actions and tools.


In [2]:
from tabulate import tabulate
from IPython.display import display, HTML

def show_trace(state):
    """Render the agent execution trace with rich formatting."""
    # 1. Try rendering rich HTML table in Jupyter
    html_rows = []
    for t in state.trace:
        tool_badge = f"<span style='background:#e0f2fe;color:#0369a1;padding:2px 8px;border-radius:12px;font-weight:600;font-size:12px;'>{t.tool}</span>" if t.tool else "<span style='color:#94a3b8;'>—</span>"
        action_badge = f"<span style='background:#f1f5f9;color:#334155;padding:2px 6px;border-radius:4px;font-weight:600;'>{t.action}</span>"
        
        inp_str = str(t.input) if t.input is not None else "—"
        res_str = str(t.result) if t.result is not None else "—"
        dec_str = str(t.decision) if t.decision else "—"
        
        html_rows.append(f"""
        <tr style="border-bottom: 1px solid #e2e8f0;">
            <td style="padding: 10px; text-align: center; font-weight: bold; color: #64748b;">{t.step}</td>
            <td style="padding: 10px;">{action_badge}</td>
            <td style="padding: 10px; text-align: center;">{tool_badge}</td>
            <td style="padding: 10px; font-family: monospace; font-size: 12px; color: #1e293b;">{inp_str}</td>
            <td style="padding: 10px; font-size: 13px; color: #0f172a;">{res_str}</td>
            <td style="padding: 10px; font-size: 13px; font-weight: 500; color: #047857;">{dec_str}</td>
        </tr>
        """)
        
    table_html = f"""
    <div style="margin: 15px 0; overflow-x: auto; border: 1px solid #cbd5e1; border-radius: 8px; box-shadow: 0 1px 3px rgba(0,0,0,0.05);">
        <table style="width: 100%; border-collapse: collapse; text-align: left; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
            <thead>
                <tr style="background: #f8fafc; border-bottom: 2px solid #cbd5e1;">
                    <th style="padding: 12px 10px; text-align: center; width: 60px; color: #475569;">Step</th>
                    <th style="padding: 12px 10px; width: 140px; color: #475569;">Action</th>
                    <th style="padding: 12px 10px; text-align: center; width: 130px; color: #475569;">Tool</th>
                    <th style="padding: 12px 10px; width: 220px; color: #475569;">Input</th>
                    <th style="padding: 12px 10px; color: #475569;">Result</th>
                    <th style="padding: 12px 10px; width: 220px; color: #475569;">Decision</th>
                </tr>
            </thead>
            <tbody>
                {"".join(html_rows)}
            </tbody>
        </table>
    </div>
    """
    display(HTML(table_html))


def show_recommendation(agent, state):
    """Display the agent's final recommendation."""
    print(agent.get_response_text(state))


def show_memory(agent):
    """Show current memory state."""
    print(f"📝 Memory State: {agent.memory.summary_str()} (Turn {agent.memory.turn_count})")

print("✅ Agent Execution Trace Helper loaded")


✅ Agent Execution Trace Helper loaded


---
## 3. Scenario 1: Successful Recommendation

**Goal:** Find a non-stop flight from Delhi to Mumbai under ₹6,000.

The agent executes an 8-step loop:
1. **`retrieve_memory`**: Checks prior preferences (empty on Turn 1).
2. **`parse_request`**: Extracts origin (`DEL`), destination (`BOM`), budget (`₹6,000`), and non-stop preference (`True`).
3. **`plan`**: Formulates ordered action plan.
4. **`tool_call (search_flights)`**: Queries dataset for `DEL → BOM`.
5. **`observe`**: Evaluates 10 options against budget ceiling (4 under budget).
6. **`tool_call (compare_price)`**: Computes ranking and differentials.
7. **`decision`**: Selects non-stop option within budget (`6E881`) and secondary fallback (`IX501`).
8. **`memory_update`**: Stores state for subsequent turns.


In [3]:
# Create a fresh agent
agent = FlightAgent()

goal = "Find a flight from Delhi to Mumbai under 6000. I prefer non-stop."
print(f"🎯 USER GOAL: \"{goal}\"\n")

# Run agent
state = agent.run(goal)

# Render formatted execution trace table
print("📊 AGENT EXECUTION TRACE:")
show_trace(state)

print("\n" + "=" * 70)
print("🤖 FINAL AGENT RESPONSE:")
print("=" * 70)
show_recommendation(agent, state)
print("-" * 70)
show_memory(agent)


🎯 USER GOAL: "Find a flight from Delhi to Mumbai under 6000. I prefer non-stop."

📊 AGENT EXECUTION TRACE:


Step,Action,Tool,Input,Result,Decision
1,retrieve_memory,memory,session (turn 1),Empty (no prior preferences),Use remembered preferences for this turn
2,parse_request,planner,Find a flight from Delhi to Mumbai under 6000. I prefer non-stop.,"{'budget': 6000.0, 'non_stop': True, 'origin': 'DEL', 'destination': 'BOM'}",Extracted user intent from natural language
3,plan,—,"{'origin': 'DEL', 'destination': 'BOM', 'budget': 6000.0, 'non_stop': True, 'preferred_time': None}","['1. search_flights(DEL → BOM)', '2. Filter by budget ≤ ₹6,000', '3. compare_price() on candidates', '4. Apply preferences: non-stop', '5. Select best + second-best']",Execute search → compare → decide
4,tool_call,search_flights,"{'from': 'DEL', 'to': 'BOM'}","10 flight(s) found (₹4,200 – ₹9,500)",Proceed to observe results
5,observe,agent,"Budget filter ≤ ₹6,000","4 under budget, 6 over budget",Compare under-budget options
6,tool_call,compare_price,"{'options_count': 4, 'budget': 6000.0}","Cheapest: IX501 (Air India Express) at ₹4,200. Second-best: SG101 (SpiceJet) at ₹4,800 (₹600 more). Under budget (₹6,000): 4 option(s), of which 2 are non-stop.",Proceed to decision engine
7,decision,agent,Scored and ranked candidates,"Best: 6E881 (IndiGo) at ₹5,200 | Second-best: IX501 (Air India Express) at ₹4,200",Selected best option based on budget + preferences
8,memory_update,memory,Write preferences and result,"Budget=₹6,000, Origin=DEL, Destination=BOM, Non-stop=Yes, Last flight=6E881",Memory updated for future turns



🤖 FINAL AGENT RESPONSE:
✅ **Recommended Flight**
   6E881 | IndiGo | DEL→BOM | 06:15–08:30 | ₹5,200 | Non-stop
   • ₹5,200 — ₹800 under your ₹6,000 budget
   • Non-stop flight

🔄 **Second-best Fallback**
   IX501 | Air India Express | ₹4,200 | 1 stop(s)
   • ₹1,000 less than recommended
----------------------------------------------------------------------
📝 Memory State: Budget=₹6,000, Origin=DEL, Destination=BOM, Non-stop=Yes, Last flight=6E881 (Turn 1)


---
## 4. Scenario 2: Memory Across Turns

This demonstrates that the agent **remembers user preferences** across multiple conversational turns without requiring them to repeat the route, budget, or preferences.

* **Turn 1**: User provides route, budget, and non-stop preference.
* **Turn 2**: User simply asks *"Find another suitable flight."* The agent retrieves and acts on stored memory.


### Turn 1: Set Preferences and Initial Search


In [4]:
agent2 = FlightAgent()

turn1_goal = "I want to fly Delhi to Mumbai under 6000. Prefer non-stop."
print(f"🎯 USER (Turn 1): \"{turn1_goal}\"\n")

state1 = agent2.run(turn1_goal)
show_trace(state1)

print("\n" + "=" * 70)
print("🤖 RESPONSE (Turn 1):")
print("=" * 70)
show_recommendation(agent2, state1)
print("-" * 70)
show_memory(agent2)


🎯 USER (Turn 1): "I want to fly Delhi to Mumbai under 6000. Prefer non-stop."



Step,Action,Tool,Input,Result,Decision
1,retrieve_memory,memory,session (turn 1),Empty (no prior preferences),Use remembered preferences for this turn
2,parse_request,planner,I want to fly Delhi to Mumbai under 6000. Prefer non-stop.,"{'budget': 6000.0, 'non_stop': True, 'origin': 'DEL', 'destination': 'BOM'}",Extracted user intent from natural language
3,plan,—,"{'origin': 'DEL', 'destination': 'BOM', 'budget': 6000.0, 'non_stop': True, 'preferred_time': None}","['1. search_flights(DEL → BOM)', '2. Filter by budget ≤ ₹6,000', '3. compare_price() on candidates', '4. Apply preferences: non-stop', '5. Select best + second-best']",Execute search → compare → decide
4,tool_call,search_flights,"{'from': 'DEL', 'to': 'BOM'}","10 flight(s) found (₹4,200 – ₹9,500)",Proceed to observe results
5,observe,agent,"Budget filter ≤ ₹6,000","4 under budget, 6 over budget",Compare under-budget options
6,tool_call,compare_price,"{'options_count': 4, 'budget': 6000.0}","Cheapest: IX501 (Air India Express) at ₹4,200. Second-best: SG101 (SpiceJet) at ₹4,800 (₹600 more). Under budget (₹6,000): 4 option(s), of which 2 are non-stop.",Proceed to decision engine
7,decision,agent,Scored and ranked candidates,"Best: 6E881 (IndiGo) at ₹5,200 | Second-best: IX501 (Air India Express) at ₹4,200",Selected best option based on budget + preferences
8,memory_update,memory,Write preferences and result,"Budget=₹6,000, Origin=DEL, Destination=BOM, Non-stop=Yes, Last flight=6E881",Memory updated for future turns



🤖 RESPONSE (Turn 1):
✅ **Recommended Flight**
   6E881 | IndiGo | DEL→BOM | 06:15–08:30 | ₹5,200 | Non-stop
   • ₹5,200 — ₹800 under your ₹6,000 budget
   • Non-stop flight

🔄 **Second-best Fallback**
   IX501 | Air India Express | ₹4,200 | 1 stop(s)
   • ₹1,000 less than recommended
----------------------------------------------------------------------
📝 Memory State: Budget=₹6,000, Origin=DEL, Destination=BOM, Non-stop=Yes, Last flight=6E881 (Turn 1)


### Turn 2: Memory-Driven Search (No constraints re-stated)


In [5]:
turn2_goal = "Find another suitable flight."
print(f"🎯 USER (Turn 2): \"{turn2_goal}\"\n")

state2 = agent2.run(turn2_goal)
show_trace(state2)

print("\n🔍 Memory Verification for Turn 2:")
print(f"   • Origin: {state2.origin} (from memory)")
print(f"   • Destination: {state2.destination} (from memory)")
print(f"   • Budget: ₹{state2.budget:,.0f} (from memory)")
print(f"   • Non-Stop Preference: {state2.non_stop} (from memory)")

print("\n" + "=" * 70)
print("🤖 RESPONSE (Turn 2):")
print("=" * 70)
show_recommendation(agent2, state2)
print("-" * 70)
show_memory(agent2)


🎯 USER (Turn 2): "Find another suitable flight."



Step,Action,Tool,Input,Result,Decision
1,retrieve_memory,memory,session (turn 2),"Budget=₹6,000, Origin=DEL, Destination=BOM, Non-stop=Yes, Last flight=6E881",Use remembered preferences for this turn
2,parse_request,planner,Find another suitable flight.,{},Extracted user intent from natural language
3,plan,—,"{'origin': 'DEL', 'destination': 'BOM', 'budget': 6000.0, 'non_stop': True, 'preferred_time': None}","['1. search_flights(DEL → BOM)', '2. Filter by budget ≤ ₹6,000', '3. compare_price() on candidates', '4. Apply preferences: non-stop', '5. Select best + second-best']",Execute search → compare → decide
4,tool_call,search_flights,"{'from': 'DEL', 'to': 'BOM'}","10 flight(s) found (₹4,200 – ₹9,500)",Proceed to observe results
5,observe,agent,"Budget filter ≤ ₹6,000","4 under budget, 6 over budget",Compare under-budget options
6,tool_call,compare_price,"{'options_count': 4, 'budget': 6000.0}","Cheapest: IX501 (Air India Express) at ₹4,200. Second-best: SG101 (SpiceJet) at ₹4,800 (₹600 more). Under budget (₹6,000): 4 option(s), of which 2 are non-stop.",Proceed to decision engine
7,decision,agent,Scored and ranked candidates,"Best: 6E881 (IndiGo) at ₹5,200 | Second-best: IX501 (Air India Express) at ₹4,200",Selected best option based on budget + preferences
8,memory_update,memory,Write preferences and result,"Budget=₹6,000, Origin=DEL, Destination=BOM, Non-stop=Yes, Last flight=6E881",Memory updated for future turns



🔍 Memory Verification for Turn 2:
   • Origin: DEL (from memory)
   • Destination: BOM (from memory)
   • Budget: ₹6,000 (from memory)
   • Non-Stop Preference: True (from memory)

🤖 RESPONSE (Turn 2):
✅ **Recommended Flight**
   6E881 | IndiGo | DEL→BOM | 06:15–08:30 | ₹5,200 | Non-stop
   • ₹5,200 — ₹800 under your ₹6,000 budget
   • Non-stop flight

🔄 **Second-best Fallback**
   IX501 | Air India Express | ₹4,200 | 1 stop(s)
   • ₹1,000 less than recommended
----------------------------------------------------------------------
📝 Memory State: Budget=₹6,000, Origin=DEL, Destination=BOM, Non-stop=Yes, Last flight=6E881 (Turn 2)


---
## 5. Scenario 3: Honest Failure (Budget Deficit Reporting)

**Goal:** Find a non-stop Delhi to Mumbai flight under ₹2,000.

No flight in the dataset costs less than ₹2,000.
The agent:
1. Calls `search_flights` and finds 10 flights.
2. Observes 0 flights under ₹2,000.
3. Invokes `compare_price` to identify the closest candidate.
4. Honestly reports that no flight fits the budget and explains the exact deficit to the cheapest available flight.


In [6]:
agent3 = FlightAgent()

goal3 = "Find a non-stop Delhi to Mumbai flight under 2000"
print(f"🎯 USER GOAL: \"{goal3}\"\n")

state3 = agent3.run(goal3)
show_trace(state3)

print("\n" + "=" * 70)
print("🤖 RESPONSE (Scenario 3 - Honest Failure):")
print("=" * 70)
show_recommendation(agent3, state3)
print("-" * 70)
show_memory(agent3)


🎯 USER GOAL: "Find a non-stop Delhi to Mumbai flight under 2000"



Step,Action,Tool,Input,Result,Decision
1,retrieve_memory,memory,session (turn 1),Empty (no prior preferences),Use remembered preferences for this turn
2,parse_request,planner,Find a non-stop Delhi to Mumbai flight under 2000,"{'budget': 2000.0, 'non_stop': True, 'origin': 'DEL', 'destination': 'BOM'}",Extracted user intent from natural language
3,plan,—,"{'origin': 'DEL', 'destination': 'BOM', 'budget': 2000.0, 'non_stop': True, 'preferred_time': None}","['1. search_flights(DEL → BOM)', '2. Filter by budget ≤ ₹2,000', '3. compare_price() on candidates', '4. Apply preferences: non-stop', '5. Select best + second-best']",Execute search → compare → decide
4,tool_call,search_flights,"{'from': 'DEL', 'to': 'BOM'}","10 flight(s) found (₹4,200 – ₹9,500)",Proceed to observe results
5,observe,agent,"Budget filter ≤ ₹2,000","0 under budget, 10 over budget",No flights under budget — compare all to find closest
6,tool_call,compare_price,"{'options_count': 10, 'budget': 2000.0}","Cheapest: IX501 (Air India Express) at ₹4,200. Second-best: SG101 (SpiceJet) at ₹4,800 (₹600 more). Under budget (₹2,000): 0 option(s), of which 0 are non-stop.",Proceed to decision engine
7,decision,agent,"Budget=₹2,000","No options under budget. Closest: ₹4,200",Report honestly — budget cannot be met
8,memory_update,memory,Write preferences and result,"Budget=₹2,000, Origin=DEL, Destination=BOM, Non-stop=Yes",Memory updated for future turns



🤖 RESPONSE (Scenario 3 - Honest Failure):
⚠️ No flight fits the ₹2,000 budget. Cheapest available: IX501 (Air India Express) at ₹4,200 — ₹2,200 over budget. Second-cheapest: SG101 (SpiceJet) at ₹4,800.

💡 Closest option: IX501 (Air India Express) — ₹4,200
----------------------------------------------------------------------
📝 Memory State: Budget=₹2,000, Origin=DEL, Destination=BOM, Non-stop=Yes (Turn 1)


---
## 6. Direct Tool Verification

Directly test the two tools to verify they function independently as computational units.


In [7]:
# 1. Tool 1: search_flights
print("═══ Tool 1: search_flights(from_city, to_city) ═══\n")
flights = search_flights("DEL", "BOM")
print(f"Found {len(flights)} flights on DEL → BOM:")
for f in flights:
    stops_str = "Non-stop" if f["stops"] == 0 else f"{f['stops']} stop(s)"
    print(f"  • {f['flight_id']:6s} | {f['airline']:20s} | {f['departure']}–{f['arrival']} | ₹{f['price']:>6,.0f} | {stops_str}")


═══ Tool 1: search_flights(from_city, to_city) ═══

Found 10 flights on DEL → BOM:
  • AI101  | Air India            | 06:00–08:05 | ₹ 7,800 | Non-stop
  • AI203  | Air India            | 08:00–10:10 | ₹ 7,450 | Non-stop
  • 6E412  | IndiGo               | 14:30–16:45 | ₹ 5,500 | Non-stop
  • UK911  | Vistara              | 09:15–11:30 | ₹ 8,200 | Non-stop
  • SG101  | SpiceJet             | 10:00–13:30 | ₹ 4,800 | 1 stop(s)
  • AI405  | Air India            | 19:00–21:15 | ₹ 7,600 | Non-stop
  • 6E881  | IndiGo               | 06:15–08:30 | ₹ 5,200 | Non-stop
  • IX501  | Air India Express    | 07:45–11:00 | ₹ 4,200 | 1 stop(s)
  • G8234  | Go First             | 15:50–18:05 | ₹ 6,950 | Non-stop
  • UK802  | Vistara              | 11:00–13:15 | ₹ 9,500 | Non-stop


In [8]:
# 2. Tool 2: compare_price
print("═══ Tool 2: compare_price(options, budget) ═══\n")
comparison = compare_price(flights, budget=6000)
print(f"Summary: {comparison['summary']}\n")
print(f"Cheapest:             {comparison['cheapest']['flight_id']} (₹{comparison['cheapest']['price']:,.0f})")
print(f"Second-Best:          {comparison['second_best']['flight_id']} (₹{comparison['second_best']['price']:,.0f})")
print(f"Price Difference:     ₹{comparison['price_difference']:,.0f}")
print(f"Under Budget:         {len(comparison['under_budget'])} option(s)")
print(f"Non-Stop Under Budget:{len(comparison['nonstop_under_budget'])} option(s)")


═══ Tool 2: compare_price(options, budget) ═══

Summary: Cheapest: IX501 (Air India Express) at ₹4,200. Second-best: SG101 (SpiceJet) at ₹4,800 (₹600 more). Under budget (₹6,000): 4 option(s), of which 2 are non-stop.

Cheapest:             IX501 (₹4,200)
Second-Best:          SG101 (₹4,800)
Price Difference:     ₹600
Under Budget:         4 option(s)
Non-Stop Under Budget:2 option(s)


In [9]:
# 3. Edge Case: Empty List Handling
empty_comp = compare_price([])
print("compare_price([]) ->", empty_comp)
print("\n✅ compare_price handles empty input safely without crashing")


compare_price([]) -> {'cheapest': None, 'second_best': None, 'under_budget': [], 'over_budget': [], 'nonstop_under_budget': [], 'price_difference': None, 'summary': 'No options available to compare.'}

✅ compare_price handles empty input safely without crashing
